In [1]:
!pip install -q --upgrade scikit-learn joblib numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 306.1/306.1 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 68.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.61.2 requires numpy<2.3,>=1.24, but you have numpy 2.5.3 which is incompatible.


In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
from pathlib import Path

# مسیر ریشه پروژه در گوگل درایو
PROJECT_ROOT = Path("/content/drive/MyDrive/CognitiveLayer/M20_6_3_3")
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
RESULT_DIR = PROJECT_ROOT / "results"
MODEL_DIR = PROJECT_ROOT / "models"

for folder in [PROJECT_ROOT, ARTIFACT_DIR, RESULT_DIR, MODEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("✅ پوشه‌ها با موفقیت ایجاد شدند یا از قبل موجود بودند:")
print(f"مسیر Artifacts: {ARTIFACT_DIR}")

✅ پوشه‌ها با موفقیت ایجاد شدند یا از قبل موجود بودند:
مسیر Artifacts: /content/drive/MyDrive/CognitiveLayer/M20_6_3_3/artifacts


In [ ]:
import shutil
from google.colab import files

print("لطفاً ۵ فایل .npz را انتخاب و آپلود کنید:")
uploaded = files.upload()

for filename in uploaded:
    src = Path("/content") / filename
    dst = ARTIFACT_DIR / filename
    shutil.copy2(src, dst)
    print(f"✅ فایل {filename} به پوشه درایو منتقل شد.")

لطفاً ۵ فایل .npz را انتخاب و آپلود کنید:


In [ ]:
required_files = [
    "TRAIN.npz",
    "SELECTION.npz",
    "VALIDATION.npz",
    "NOVELTY_TRAIN.npz",
    "NOVELTY_TEST.npz",
]

all_ready = True
for fname in required_files:
    fpath = ARTIFACT_DIR / fname
    if fpath.exists():
        size_mb = fpath.stat().st_size / (1024 * 1024)
        print(f"✅ {fname:20s} [موجود - {size_mb:.2f} MB]")
    else:
        print(f"❌ {fname:20s} [یافت نشد!]")
        all_ready = False

if all_ready:
    print("\n🚀 تمام پیش‌نیازها آماده است. اکنون می‌توانید کد ارزیابی اصلی را اجرا کنید.")
else:
    print("\n⚠️ خطا: برخی فایل‌ها در مسیر Drive وجود ندارند. ابتدا آن‌ها را تأمین کنید.")

In [ ]:
# ==============================================================================
# M20.6.3.3 — Endogenous Self-State Incremental-Value Audit (Colab Unified Code)
# ==============================================================================

# ------------------------------------------------------------------------------
# 1. نصب و ارتقای پیش‌نیازها
# ------------------------------------------------------------------------------
!pip install -q --upgrade scikit-learn joblib

# ------------------------------------------------------------------------------
# 2. ایمپورت ماژول‌ها و تنظیمات محیطی
# ------------------------------------------------------------------------------
import json
import math
import os
import platform
import shutil
import sys
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, Optional, Tuple

import joblib
import numpy as np
import sklearn
from sklearn.decomposition import PCA
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    mean_absolute_error,
    r2_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

print(f"Python: {sys.version.split()[0]} | NumPy: {np.__version__} | scikit-learn: {sklearn.__version__}")

# ------------------------------------------------------------------------------
# 3. اتصال Google Drive
# ------------------------------------------------------------------------------
from google.colab import drive
drive.mount("/content/drive")

# ------------------------------------------------------------------------------
# 4. پیکربندی (Config) و مسیرها
# ------------------------------------------------------------------------------
@dataclass(frozen=True)
class Config:
    seed: int = 206333
    project_root: str = "/content/drive/MyDrive/CognitiveLayer/M20_6_3_3"
    artifact_subdirectory: str = "artifacts"
    result_subdirectory: str = "results"
    model_subdirectory: str = "models"

    # Representation
    activation_pca_dim: int = 12
    activation_residualizer_alpha: float = 10.0

    # Model Grids
    ridge_alphas: Tuple[float, ...] = (0.01, 0.1, 1.0, 10.0, 100.0)
    logistic_cs: Tuple[float, ...] = (0.01, 0.1, 1.0, 10.0)

    # Statistical Evaluation (برای تست سریع می‌توانید این‌ها را کاهش دهید)
    n_bootstrap: int = 1000
    n_permutations: int = 500
    n_temporal_permutations: int = 300

    # Threshold Gates
    point_delta_r2_minimum: float = 0.01
    hazard_delta_auc_minimum: float = 0.01
    hazard_delta_ap_minimum: float = 0.02
    hazard_brier_improvement_minimum: float = 0.005
    temporal_point_margin: float = 0.02
    temporal_hazard_margin: float = 0.02
    novelty_auc_minimum: float = 0.80
    novelty_ap_above_prevalence_minimum: float = 0.15

    # Operational Policy
    event_max_lead: int = 3
    policy_minimum_event_recall: float = 0.60
    policy_maximum_false_episodes_per_100: float = 3.0
    threshold_grid_size: int = 199

CFG = Config()
PROJECT_ROOT = Path(CFG.project_root)
ARTIFACT_DIR = PROJECT_ROOT / CFG.artifact_subdirectory
RESULT_DIR = PROJECT_ROOT / CFG.result_subdirectory
MODEL_DIR = PROJECT_ROOT / CFG.model_subdirectory

for directory in [PROJECT_ROOT, ARTIFACT_DIR, RESULT_DIR, MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

DATA_PATHS = {
    "train": ARTIFACT_DIR / "TRAIN.npz",
    "selection": ARTIFACT_DIR / "SELECTION.npz",
    "validation": ARTIFACT_DIR / "VALIDATION.npz",
    "novelty_train": ARTIFACT_DIR / "NOVELTY_TRAIN.npz",
    "novelty_test": ARTIFACT_DIR / "NOVELTY_TEST.npz",
}

REPORT_PATH = RESULT_DIR / "m20_6_3_3_endogenous_self_state_audit.json"
MODEL_BUNDLE_PATH = MODEL_DIR / "m20_6_3_3_model_bundle.joblib"

# ------------------------------------------------------------------------------
# 5. بررسی وجود فایل‌های ورودی (.npz)
# ------------------------------------------------------------------------------
missing_files = []
for split_name, path in DATA_PATHS.items():
    if path.exists():
        size_mb = path.stat().st_size / (1024 ** 2)
        print(f"✅ {split_name:16s} {path.name:22s} {size_mb:8.2f} MB")
    else:
        print(f"❌ {split_name:16s} {path.name:22s} MISSING")
        missing_files.append(str(path))

if missing_files:
    raise FileNotFoundError(
        "فایل‌های زیر یافت نشدند. ابتدا آن‌ها را در مسیر Drive قرار دهید:\n"
        + "\n".join(missing_files)
    )

# ------------------------------------------------------------------------------
# 6. ساختار داده، لودر و متدهای آماری
# ------------------------------------------------------------------------------
@dataclass
class Dataset:
    X_raw: np.ndarray
    X_activation: np.ndarray
    y_point: np.ndarray
    y_hazard: np.ndarray
    stream_id: np.ndarray
    time_index: np.ndarray
    novelty_label: Optional[np.ndarray] = None

    def validate(self, name: str, require_novelty: bool = False) -> None:
        if self.X_raw.ndim != 3 or self.X_activation.ndim != 3:
            raise ValueError(f"{name}: Both X_raw and X_activation must be 3D [N, H, D].")
        n = self.X_raw.shape[0]
        for field, val in {"X_act": self.X_activation, "y_pt": self.y_point, "y_hz": self.y_hazard}.items():
            if len(val) != n:
                raise ValueError(f"{name}: {field} length mismatch.")
        if self.X_raw.shape[1] != self.X_activation.shape[1] or self.X_raw.shape[1] < 3:
            raise ValueError(f"{name}: Sequence length error (must match and be >= 3).")
        if require_novelty and self.novelty_label is None:
            raise ValueError(f"{name}: novelty_label is required.")

def load_dataset(path: Path, name: str, require_novelty: bool = False) -> Dataset:
    with np.load(path, allow_pickle=True) as p:
        novelty_lbl = np.asarray(p["novelty_label"], dtype=np.int64).reshape(-1) if "novelty_label" in p.files else None
        ds = Dataset(
            X_raw=np.asarray(p["X_raw"], dtype=np.float32),
            X_activation=np.asarray(p["X_activation"], dtype=np.float32),
            y_point=np.asarray(p["y_point"], dtype=np.float64).reshape(-1),
            y_hazard=np.asarray(p["y_hazard"], dtype=np.int64).reshape(-1),
            stream_id=np.asarray(p["stream_id"]).reshape(-1),
            time_index=np.asarray(p["time_index"], dtype=np.int64).reshape(-1),
            novelty_label=novelty_lbl,
        )
    ds.validate(name, require_novelty)
    return ds

train = load_dataset(DATA_PATHS["train"], "TRAIN")
selection = load_dataset(DATA_PATHS["selection"], "SELECTION")
validation = load_dataset(DATA_PATHS["validation"], "VALIDATION")
novelty_train = load_dataset(DATA_PATHS["novelty_train"], "NOVELTY_TRAIN", require_novelty=True)
novelty_test = load_dataset(DATA_PATHS["novelty_test"], "NOVELTY_TEST", require_novelty=True)

# توابع کمکی مانیتورینگ و ارزیابی
def safe_pearson(y_true, y_pred):
    if len(y_true) < 2 or np.std(y_true) < 1e-12 or np.std(y_pred) < 1e-12:
        return float("nan")
    return float(np.corrcoef(y_true, y_pred)[0, 1])

def safe_auc(y_true, prob):
    return float(roc_auc_score(y_true, prob)) if len(np.unique(y_true)) >= 2 else float("nan")

def safe_ap(y_true, prob):
    return float(average_precision_score(y_true, prob)) if np.sum(y_true == 1) > 0 else float("nan")

def point_metrics(y_true, pred):
    return {
        "rho": safe_pearson(y_true, pred),
        "r2": float(r2_score(y_true, pred)),
        "mae": float(mean_absolute_error(y_true, pred)),
    }

def hazard_metrics(y_true, prob):
    return {
        "auc": safe_auc(y_true, prob),
        "ap": safe_ap(y_true, prob),
        "brier": float(brier_score_loss(y_true, prob)),
        "prevalence": float(np.mean(y_true)),
    }

def flatten_sequence(seq):
    return seq.reshape(len(seq), -1)

def to_json_safe(v):
    if isinstance(v, dict):
        return {str(k): to_json_safe(val) for k, val in v.items()}
    if isinstance(v, (list, tuple)):
        return [to_json_safe(x) for x in v]
    if isinstance(v, np.ndarray):
        return to_json_safe(v.tolist())
    if isinstance(v, (np.bool_, bool)):
        return bool(v)
    if isinstance(v, (np.integer, int)):
        return int(v)
    if isinstance(v, (np.floating, float)):
        return float(v) if np.isfinite(v) else None
    return v

# ------------------------------------------------------------------------------
# 7. ساخت Representation و Residualization
# ------------------------------------------------------------------------------
class ActivationRepresentation:
    def __init__(self, pca_dim: int, seed: int):
        self.pca_dim = int(pca_dim)
        self.seed = int(seed)
        self.scaler = StandardScaler()
        self.pca = None

    def fit(self, dataset: Dataset):
        rows = dataset.X_activation.reshape(-1, dataset.X_activation.shape[-1])
        scaled = self.scaler.fit_transform(rows)
        max_dim = min(self.pca_dim, scaled.shape[1], max(1, scaled.shape[0] - 1))
        self.pca = PCA(n_components=max_dim, random_state=self.seed)
        self.pca.fit(scaled)
        return self

    def transform_sequence(self, act: np.ndarray):
        n, hist, dim = act.shape
        rows = self.pca.transform(self.scaler.transform(act.reshape(-1, dim)))
        return rows.reshape(n, hist, -1)

    def transform(self, dataset: Dataset):
        return flatten_sequence(self.transform_sequence(dataset.X_activation))

class ActivationResidualizer:
    def __init__(self, alpha: float):
        self.model = Pipeline([("scale", StandardScaler()), ("ridge", Ridge(alpha=alpha))])

    def fit(self, X_raw, X_act):
        self.model.fit(X_raw, X_act)
        return self

    def transform(self, X_raw, X_act):
        return X_act - self.model.predict(X_raw)

# برازش مدل‌های بازنمایی روی TRAIN
act_rep = ActivationRepresentation(CFG.activation_pca_dim, CFG.seed).fit(train)
raw_train, raw_sel, raw_val = flatten_sequence(train.X_raw), flatten_sequence(selection.X_raw), flatten_sequence(validation.X_raw)
act_train, act_sel, act_val = act_rep.transform(train), act_rep.transform(selection), act_rep.transform(validation)

residualizer = ActivationResidualizer(CFG.activation_residualizer_alpha).fit(raw_train, act_train)
res_train, res_sel, res_val = residualizer.transform(raw_train, act_train), residualizer.transform(raw_sel, act_sel), residualizer.transform(raw_val, act_val)

comb_train = np.concatenate([raw_train, res_train], axis=1)
comb_sel = np.concatenate([raw_sel, res_sel], axis=1)
comb_val = np.concatenate([raw_val, res_val], axis=1)

# ------------------------------------------------------------------------------
# 8. انتخاب هایپرپارامترها بر روی SELECTION
# ------------------------------------------------------------------------------
def select_point_model(X_tr, y_tr, X_s, y_s):
    best_m, best_rec, best_sc = None, None, -np.inf
    for alpha in CFG.ridge_alphas:
        m = Pipeline([("scale", StandardScaler()), ("ridge", Ridge(alpha=alpha))]).fit(X_tr, y_tr)
        metrics = point_metrics(y_s, m.predict(X_s))
        sc = metrics["r2"] + 0.25 * metrics["rho"]
        if np.isfinite(sc) and sc > best_sc:
            best_sc, best_m, best_rec = sc, m, {"alpha": float(alpha), "metrics": metrics}
    return best_m, best_rec

def select_hazard_model(X_tr, y_tr, X_s, y_s):
    best_m, best_rec, best_sc = None, None, -np.inf
    for c in CFG.logistic_cs:
        m = Pipeline([
            ("scale", StandardScaler()),
            ("logistic", LogisticRegression(C=c, max_iter=5000, class_weight="balanced", random_state=CFG.seed))
        ]).fit(X_tr, y_tr)
        metrics = hazard_metrics(y_s, m.predict_proba(X_s)[:, 1])
        sc = metrics["auc"] + 0.5 * metrics["ap"] - 0.25 * metrics["brier"]
        if np.isfinite(sc) and sc > best_sc:
            best_sc, best_m, best_rec = sc, m, {"C": float(c), "metrics": metrics}
    return best_m, best_rec

raw_pt_model, raw_pt_sel = select_point_model(raw_train, train.y_point, raw_sel, selection.y_point)
comb_pt_model, comb_pt_sel = select_point_model(comb_train, train.y_point, comb_sel, selection.y_point)
raw_hz_model, raw_hz_sel = select_hazard_model(raw_train, train.y_hazard, raw_sel, selection.y_hazard)
comb_hz_model, comb_hz_sel = select_hazard_model(comb_train, train.y_hazard, comb_sel, selection.y_hazard)

# ------------------------------------------------------------------------------
# 9. ارزیابی نهایی روی VALIDATION
# ------------------------------------------------------------------------------
raw_pt_pred = raw_pt_model.predict(raw_val)
comb_pt_pred = comb_pt_model.predict(comb_val)
raw_hz_prob = raw_hz_model.predict_proba(raw_val)[:, 1]
comb_hz_prob = comb_hz_model.predict_proba(comb_val)[:, 1]

raw_pt_res = point_metrics(validation.y_point, raw_pt_pred)
comb_pt_res = point_metrics(validation.y_point, comb_pt_pred)
raw_hz_res = hazard_metrics(validation.y_hazard, raw_hz_prob)
comb_hz_res = hazard_metrics(validation.y_hazard, comb_hz_prob)

point_delta_r2 = comb_pt_res["r2"] - raw_pt_res["r2"]
hazard_delta_auc = comb_hz_res["auc"] - raw_hz_res["auc"]
hazard_delta_ap = comb_hz_res["ap"] - raw_hz_res["ap"]
hazard_brier_imp = raw_hz_res["brier"] - comb_hz_res["brier"]

# ------------------------------------------------------------------------------
# 10. ارزیابی آماری: Bootstrap & Permutation
# ------------------------------------------------------------------------------
def stream_bootstrap(stream_id, y_true, base_pred, aug_pred, score_fn, n_iter, seed):
    rng = np.random.default_rng(seed)
    unique_streams = np.unique(stream_id)
    obs = score_fn(y_true, aug_pred) - score_fn(y_true, base_pred)
    values = []
    for _ in range(n_iter):
        sampled = rng.choice(unique_streams, size=len(unique_streams), replace=True)
        idx = np.concatenate([np.flatnonzero(stream_id == s) for s in sampled])
        val = score_fn(y_true[idx], aug_pred[idx]) - score_fn(y_true[idx], base_pred[idx])
        if np.isfinite(val):
            values.append(float(val))
    return {
        "observed": float(obs),
        "ci95": [float(np.percentile(values, 2.5)), float(np.percentile(values, 97.5))],
        "mean": float(np.mean(values))
    }

point_r2_boot = stream_bootstrap(validation.stream_id, validation.y_point, raw_pt_pred, comb_pt_pred, r2_score, CFG.n_bootstrap, CFG.seed + 1)
hazard_auc_boot = stream_bootstrap(validation.stream_id, validation.y_hazard, raw_hz_prob, comb_hz_prob, safe_auc, CFG.n_bootstrap, CFG.seed + 2)
hazard_ap_boot = stream_bootstrap(validation.stream_id, validation.y_hazard, raw_hz_prob, comb_hz_prob, safe_ap, CFG.n_bootstrap, CFG.seed + 3)
hazard_brier_boot = stream_bootstrap(validation.stream_id, validation.y_hazard, raw_hz_prob, comb_hz_prob, lambda y, p: -brier_score_loss(y, p), CFG.n_bootstrap, CFG.seed + 4)

def permutation_test(base_pred, aug_m, X_raw, X_act_res, y_true, stream_id, score_fn, is_cls, n_iter, seed):
    rng = np.random.default_rng(seed)
    obs_feat = np.concatenate([X_raw, X_act_res], axis=1)
    obs_pred = aug_m.predict_proba(obs_feat)[:, 1] if is_cls else aug_m.predict(obs_feat)
    base_sc = score_fn(y_true, base_pred)
    obs_imp = score_fn(y_true, obs_pred) - base_sc

    null_imps = []
    for _ in range(n_iter):
        perm_res = np.array(X_act_res, copy=True)
        for s in np.unique(stream_id):
            idx = np.flatnonzero(stream_id == s)
            if len(idx) > 1:
                perm_res[idx] = X_act_res[rng.permutation(idx)]
        f = np.concatenate([X_raw, perm_res], axis=1)
        pred = aug_m.predict_proba(f)[:, 1] if is_cls else aug_m.predict(f)
        null_imps.append(score_fn(y_true, pred) - base_sc)

    null_imps = np.asarray(null_imps)
    p_val = (1 + np.sum(null_imps >= obs_imp)) / (1 + len(null_imps))
    return {"observed": float(obs_imp), "p_value": float(p_val)}

pt_perm = permutation_test(raw_pt_pred, comb_pt_model, raw_val, res_val, validation.y_point, validation.stream_id, r2_score, False, CFG.n_permutations, CFG.seed + 10)
hz_auc_perm = permutation_test(raw_hz_prob, comb_hz_model, raw_val, res_val, validation.y_hazard, validation.stream_id, safe_auc, True, CFG.n_permutations, CFG.seed + 11)
hz_ap_perm = permutation_test(raw_hz_prob, comb_hz_model, raw_val, res_val, validation.y_hazard, validation.stream_id, safe_ap, True, CFG.n_permutations, CFG.seed + 12)
hz_brier_perm = permutation_test(raw_hz_prob, comb_hz_model, raw_val, res_val, validation.y_hazard, validation.stream_id, lambda y, p: -brier_score_loss(y, p), True, CFG.n_permutations, CFG.seed + 13)

# ------------------------------------------------------------------------------
# 11. آزمون Corrected Temporal-Order Ablation
# ------------------------------------------------------------------------------
def temporal_transform(seq, mode, rng):
    out = np.array(seq, copy=True)
    hist = seq[:, :-1, :]
    if mode == "REVERSED":
        out[:, :-1, :] = hist[:, ::-1, :]
    elif mode == "SHUFFLED":
        for i in range(len(out)):
            out[i, :-1, :] = hist[i, rng.permutation(hist.shape[1]), :]
    elif mode == "CIRCULAR":
        for i in range(len(out)):
            out[i, :-1, :] = np.roll(hist[i], int(rng.integers(1, hist.shape[1])), axis=0)
    return out

def run_temporal_ablation(pt_m, hz_m, ds, n_iter, seed):
    rng = np.random.default_rng(seed)
    ord_f = flatten_sequence(ds.X_raw)
    ord_pt, ord_hz = safe_pearson(ds.y_point, pt_m.predict(ord_f)), safe_auc(ds.y_hazard, hz_m.predict_proba(ord_f)[:, 1])

    rev_f = flatten_sequence(temporal_transform(ds.X_raw, "REVERSED", rng))
    rev_pt, rev_hz = safe_pearson(ds.y_point, pt_m.predict(rev_f)), safe_auc(ds.y_hazard, hz_m.predict_proba(rev_f)[:, 1])

    shuf_pt, shuf_hz, circ_pt, circ_hz = [], [], [], []
    for _ in range(n_iter):
        s_f = flatten_sequence(temporal_transform(ds.X_raw, "SHUFFLED", rng))
        shuf_pt.append(safe_pearson(ds.y_point, pt_m.predict(s_f)))
        shuf_hz.append(safe_auc(ds.y_hazard, hz_m.predict_proba(s_f)[:, 1]))
        c_f = flatten_sequence(temporal_transform(ds.X_raw, "CIRCULAR", rng))
        circ_pt.append(safe_pearson(ds.y_point, pt_m.predict(c_f)))
        circ_hz.append(safe_auc(ds.y_hazard, hz_m.predict_proba(c_f)[:, 1]))

    p_pass = bool(ord_pt >= np.nanpercentile(shuf_pt, 95) + CFG.temporal_point_margin and ord_pt > rev_pt)
    h_pass = bool(ord_hz >= np.nanpercentile(shuf_hz, 95) + CFG.temporal_hazard_margin and ord_hz > rev_hz)
    return {
        "point": {"ordered_rho": ord_pt, "reversed_rho": rev_pt, "pass": p_pass},
        "hazard": {"ordered_auc": ord_hz, "reversed_auc": rev_hz, "pass": h_pass}
    }

temporal_res = run_temporal_ablation(raw_pt_model, raw_hz_model, validation, CFG.n_temporal_permutations, CFG.seed + 20)

# ------------------------------------------------------------------------------
# 12. آزمون Novelty Structural-Identifiability Oracle
# ------------------------------------------------------------------------------
def extract_novelty_feat(ds, rep):
    raw = ds.X_raw
    f_raw = np.concatenate([
        flatten_sequence(raw), raw[:, -1, :], raw[:, -1, :] - raw[:, -2, :],
        raw[:, -1, :] - 2.0 * raw[:, -2, :] + raw[:, -3, :],
        np.mean(raw, axis=1), np.std(raw, axis=1)
    ], axis=1)
    act_seq = rep.transform_sequence(ds.X_activation)
    return np.concatenate([
        f_raw, act_seq[:, -1, :], act_seq.mean(axis=1), act_seq.std(axis=1),
        act_seq[:, -1, :] - act_seq[:, -2, :]
    ], axis=1)

X_nov_train = extract_novelty_feat(novelty_train, act_rep)
X_nov_test = extract_novelty_feat(novelty_test, act_rep)

novelty_oracle = ExtraTreesClassifier(
    n_estimators=600, min_samples_leaf=3, max_features="sqrt", class_weight="balanced", random_state=CFG.seed, n_jobs=-1
).fit(X_nov_train, novelty_train.novelty_label)

nov_prob = novelty_oracle.predict_proba(X_nov_test)[:, 1]
nov_auc = safe_auc(novelty_test.novelty_label, nov_prob)
nov_ap = safe_ap(novelty_test.novelty_label, nov_prob)
nov_prev = float(novelty_test.novelty_label.mean())
nov_pass = bool(nov_auc >= CFG.novelty_auc_minimum and (nov_ap - nov_prev) >= CFG.novelty_ap_above_prevalence_minimum)
novelty_res = {"auc": nov_auc, "ap": nov_ap, "prevalence": nov_prev, "pass": nov_pass}

# ------------------------------------------------------------------------------
# 13. ارزیابی Operational Policy Feasibility
# ------------------------------------------------------------------------------
def evaluate_policy(ds, prob):
    thresholds = np.linspace(0.01, 0.99, CFG.threshold_grid_size)
    feasible = False
    for th in thresholds:
        alert = prob >= th
        # محاسبه onsetهای رخداد
        detected, total_onsets = 0, 0
        for s in np.unique(ds.stream_id):
            idx = np.flatnonzero(ds.stream_id == s)
            idx = idx[np.argsort(ds.time_index[idx])]
            for i, c_idx in enumerate(idx):
                if ds.y_hazard[c_idx] == 1 and (i == 0 or ds.y_hazard[idx[i-1]] == 0):
                    total_onsets += 1
                    onset_t = ds.time_index[c_idx]
                    cand = idx[(ds.time_index[idx] >= onset_t - CFG.event_max_lead) & (ds.time_index[idx] <= onset_t)]
                    if np.any(alert[cand]):
                        detected += 1
        recall = detected / total_onsets if total_onsets else 0
        false_episodes = np.sum(alert & (ds.y_hazard == 0))
        false_per_100 = 100.0 * false_episodes / len(alert)
        if recall >= CFG.policy_minimum_event_recall and false_per_100 <= CFG.policy_maximum_false_episodes_per_100:
            feasible = True
            break
    return {"feasible": feasible}

raw_pol = evaluate_policy(validation, raw_hz_prob)
comb_pol = evaluate_policy(validation, comb_hz_prob)

# ------------------------------------------------------------------------------
# 14. بررسی نهایی Gateها و تشخیص (Final Diagnosis)
# ------------------------------------------------------------------------------
pt_pass = bool(point_delta_r2 >= CFG.point_delta_r2_minimum and point_r2_boot["ci95"][0] > 0 and pt_perm["p_value"] <= 0.05)
hz_auc_pass = bool(hazard_delta_auc >= CFG.hazard_delta_auc_minimum and hazard_auc_boot["ci95"][0] > 0 and hz_auc_perm["p_value"] <= 0.05)
hz_ap_pass = bool(hazard_delta_ap >= CFG.hazard_delta_ap_minimum and hazard_ap_boot["ci95"][0] > 0 and hz_ap_perm["p_value"] <= 0.05)
hz_brier_pass = bool(hazard_brier_imp >= CFG.hazard_brier_improvement_minimum and hazard_brier_boot["ci95"][0] > 0 and hz_brier_perm["p_value"] <= 0.05)
hz_pass = bool(hz_auc_pass or hz_ap_pass or hz_brier_pass)

endogenous_supported = bool(pt_pass or hz_pass)
temporal_supported = bool(temporal_res["point"]["pass"] and temporal_res["hazard"]["pass"])
overall_pass = bool(endogenous_supported and temporal_supported and novelty_res["pass"])

if not endogenous_supported:
    final_diag = "NO_INCREMENTAL_ENDOGENOUS_SIGNAL"
elif not temporal_supported:
    final_diag = "ENDOGENOUS_SIGNAL_WITHOUT_STRICT_TEMPORAL_ORDER"
elif not novelty_res["pass"]:
    final_diag = "ENDOGENOUS_TEMPORAL_STATE_SUPPORTED_BUT_NOVELTY_NOT_STRUCTURALLY_IDENTIFIABLE"
elif not (raw_pol["feasible"] or comb_pol["feasible"]):
    final_diag = "SELF_STATE_SUPPORTED_BUT_OPERATIONAL_POLICY_REQUIRES_ONSET_SPECIFIC_MODEL"
else:
    final_diag = "READY_FOR_ACTION_CONDITIONED_CAUSAL_SELF_MODEL"

# ------------------------------------------------------------------------------
# 15. ذخیره‌سازی گزارش و Model Bundle
# ------------------------------------------------------------------------------
report = {
    "milestone": "20.6.3.3",
    "final_diagnosis": final_diag,
    "statuses": {
        "endogenous_supported": endogenous_supported,
        "temporal_order_supported": temporal_supported,
        "novelty_identifiable": novelty_res["pass"],
        "policy_feasible": comb_pol["feasible"],
        "overall_pass": overall_pass,
    },
    "metrics": {
        "point_delta_r2": point_delta_r2,
        "hazard_delta_auc": hazard_delta_auc,
        "hazard_delta_ap": hazard_delta_ap,
        "hazard_brier_improvement": hazard_brier_imp
    }
}

with open(REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(to_json_safe(report), f, indent=2)

model_bundle = {
    "milestone": "20.6.3.3",
    "config": CFG,
    "activation_representation": act_rep,
    "activation_residualizer": residualizer,
    "raw_point_model": raw_pt_model,
    "combined_point_model": comb_pt_model,
    "raw_hazard_model": raw_hz_model,
    "combined_hazard_model": comb_hz_model,
    "novelty_oracle": novelty_oracle,
    "final_diagnosis": final_diag,
}

joblib.dump(model_bundle, MODEL_BUNDLE_PATH, compress=3)

print("=" * 80)
print(f"✅ AUDIT COMPLETED. Final Diagnosis: {final_diag}")
print(f"Report saved to: {REPORT_PATH}")
print(f"Bundle saved to: {MODEL_BUNDLE_PATH}")
print("=" * 80)